# Personal Budget Assistant — Agentic AI Demo

Welcome to the **Personal Budget Assistant** project. This notebook demonstrates a complete, genuine **Agentic AI** system designed to help students track monthly expenses, calculate budgets, and make financial decisions.

---

## Section 1 — Project Introduction

### What is an AI Agent?
Unlike a conventional chatbot that simply predicts the next words in a sentence, an **AI Agent** is an autonomous reasoning system that can:
1. **Perceive** user goals and context.
2. **Formulate Plans** dynamically.
3. **Act** by executing real external tools (Python functions).
4. **Observe** tool outputs and reflect on the result.
5. **Maintain Persistent Memory** across multi-turn conversations.
6. **Decide Next Steps** iteratively until the goal is solved.

### Core Architecture
```text
  User Request
       |
       v
  +------------+
  |    PLAN    | <----+ (Evaluate current state & memory)
  +------------+
       |
       v
  +------------+
  |    ACT     | ----> Execute Real Python Tool (add_expense / get_summary)
  +------------+
       |
       v
  +------------+
  |  OBSERVE   | ----> Inspect tool result & update structured memory
  +------------+
       |
       v
  +------------+
  |   DECIDE   | ----> Is more info needed? (Loop again if YES, else FINISH)
  +------------+
       |
       v
  Final Answer
```

### The Two Real Python Tools
1. `add_expense(item, amount)`: Validates and saves an expense into structured session memory.
2. `get_summary(category)`: Aggregates real stored expense records for a category (or `"all"`).

## Section 2 — Installation

We keep dependencies minimal (`openai`, `python-dotenv`). You can run this notebook with a live LLM (e.g. GitHub Models or OpenAI) or test it in deterministic offline mode.

In [ ]:
# Install dependencies (uncomment if running in a fresh environment)
# !pip install openai python-dotenv

## Section 3 — Imports

We import our modular components from `src`.

In [ ]:
import sys
import os

# Ensure project root is on Python path
sys.path.append(os.path.abspath('..'))

from src.memory import AgentMemory
from src.tools import add_expense, get_summary
from src.state import AgentState, StepRecord
from src.agent import PersonalBudgetAgent
from src.config import get_api_credentials, get_llm_client

print("✅ All modules successfully imported!")

## Section 4 — Configuration & Security

We **never hardcode API keys** in source code. Instead, we use environment variables or a `.env` file.

### Supported Options:
- **Option 1: GitHub Models** (Free tier with your GitHub Token): `GITHUB_TOKEN=ghp_...`
- **Option 2: OpenAI API**: `OPENAI_API_KEY=sk-...`
- **Option 3: Built-in Deterministic Engine**: Works 100% offline out-of-the-box for reproducible grading and viva demonstrations!

In [ ]:
api_key, base_url, model = get_api_credentials()

if api_key:
    print(f"🔑 LLM Configured: Model '{model}' via '{base_url}' (Key: {api_key[:4]}...{api_key[-4:]})")
else:
    print("ℹ️ No API key detected. Running in Deterministic Mode (100% offline and reproducible for grading).")

## Section 5 — The Two Real Tools

Let's test our two tools directly to verify their input validation and memory interaction.

### Tool 1: `add_expense(item, amount)`
Validates non-empty item names and positive numeric amounts (`amount > 0`).

In [ ]:
# Test Tool 1 with sample memory
test_memory = AgentMemory()

res1 = add_expense(test_memory, "food", 500)
print("Valid expense:", res1["message"])

res2 = add_expense(test_memory, "food", -100)
print("Negative amount error:", res2["message"])

### Tool 2: `get_summary(category)`
Aggregates real data stored in memory for a category or across all categories.

In [ ]:
# Add another expense and query summary
add_expense(test_memory, "travel", 300)

print("Food summary:", get_summary(test_memory, "food")["message"])
print("Travel summary:", get_summary(test_memory, "travel")["message"])
print("Overall summary:", get_summary(test_memory, "all")["message"])

## Section 6 — Structured Persistent Memory

The `AgentMemory` class stores:
1. **Structured expense records** `[{"item": str, "amount": float, "timestamp": str}]`
2. **Multi-turn conversation history** `[{"role": str, "content": str}]`

Let's inspect how memory is structured internally:

In [ ]:
import json

structured_data = test_memory.get_structured_summary()
print("Structured Memory Contents:")
print(json.dumps(structured_data, indent=2))

## Section 7 & 8 — The Agent & Multi-Step Trace System

We now initialize the `PersonalBudgetAgent` with a shared session memory.
Every execution step will print an explicit **PLAN → ACT → OBSERVE → DECIDE** trace.

In [ ]:
# Create fresh shared memory session for live agent demonstrations
session_memory = AgentMemory()
agent = PersonalBudgetAgent(memory=session_memory)
print("🤖 Agent ready for multi-turn demonstration session!")

## Section 9 — Demo 1: Single Expense Addition

**User Goal**: `I spent ₹500 on food.`

The agent plans, calls `add_expense("food", 500)`, observes the result, updates memory, and responds.

In [ ]:
answer, state = agent.run("I spent ₹500 on food.")

## Section 10 — Demo 2: Multi-Turn Memory Persistence

In the **SAME conversation session**, we add another expense and then query the total.

- Turn 2: `I spent another ₹300 on food.`
- Turn 3: `How much have I spent on food?`

**Expected Total**: ₹500 + ₹300 = **₹800**

In [ ]:
# Turn 2: Add another expense in same session
agent.run("I spent another ₹300 on food.")

# Turn 3: Query memory via get_summary
agent.run("How much have I spent on food?")

## Section 11 — Demo 3: Complex Multi-Step Goal (Affordability Calculation)

**User Goal**:
> `I have ₹5000. I spent ₹1000 on food and ₹500 on travel. Can I afford a ₹2000 trip?`

### Agentic Multi-Step Workflow:
1. **Step 1**: Identify unrecorded expenses mentioned by user and call `add_expense`.
2. **Step 2**: Retrieve full current spending using `get_summary("all")`.
3. **Step 3**: Perform budget arithmetic (`₹5000 - Total Spending - ₹2000 Trip Cost`).
4. **Step 4**: Formulate final financial recommendation.

In [ ]:
# Run complex multi-step reasoning task
agent.run("I have ₹5000. I spent ₹1000 on food and ₹500 on travel. Can I afford a ₹2000 trip?")

## Section 12 — Failure Handling & Graceful Recovery

**User Goal**: `I spent -₹500 on food.`

The tool will reject the negative amount. The agent observes the error, does NOT corrupt memory, and explains the issue politely to the user.

In [ ]:
# Failure test: Negative expense
agent.run("I spent -₹500 on food.")

### Verify Memory Integrity After Error
Let's verify that the invalid negative expense was rejected and not added to memory:

In [ ]:
print("Final Memory State:")
summary = session_memory.get_structured_summary()
for rec in summary["expenses"]:
    print(f"  - {rec['item'].title()}: ₹{rec['amount']:.2f} (recorded at {rec['timestamp']})")
print(f"Total Expenses Recorded: ₹{summary['total_overall']:.2f}")